# 🛡️ Multilingual Abuse Detection
## English · Hinglish · Banglish | Binary Classification + Language ID | XLM-RoBERTa-Large

---
| Section | Description |
|---|---|
| 1 | Install & imports |
| 2 | Config — all hyperparams |
| 3 | Drive mount + folder structure |
| 4 | Checkpoint Manager |
| 5 | Data upload + balanced merging |
| 6 | EDA |
| 7 | Preprocessing |
| 8 | Train / Val split |
| 9 | Dataset + DataLoaders |
| 10 | Multi-task model (XLM-R Large) |
| 11 | Training loop |
| 12 | Load best model |
| 13 | Threshold tuning |
| 14 | Full evaluation suite |
| 15 | Attention visualization |
| 16 | Interactive testing cell |


## Section 1 — Install

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 1 — INSTALL
# ═══════════════════════════════════════════════════════════
!pip install transformers torch datasets emoji langdetect -q
!pip install scikit-learn matplotlib seaborn tqdm ipywidgets -q
print("All packages installed ✅")


## Section 2 — Config *(edit only this cell)*

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 2 — CONFIG
# All hyperparameters and paths live here.
# ═══════════════════════════════════════════════════════════
import os, json, re, warnings, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Model ─────────────────────────────────────────────────
MODEL_NAME          = "xlm-roberta-large"
MAX_LEN             = 128
NUM_LANGUAGES       = 3      # english, hinglish, banglish

# ── Training ──────────────────────────────────────────────
BATCH_SIZE          = 8      # safe for XLM-R-Large on T4
GRAD_ACCUM_STEPS    = 4      # effective batch = 8×4 = 32
EPOCHS              = 10
LR                  = 1e-5   # lower LR suits the large model
WEIGHT_DECAY        = 0.01
WARMUP_RATIO        = 0.1
EARLY_STOP_PATIENCE = 5

# ── Multi-task loss weights ────────────────────────────────
OFFENSIVE_LOSS_W    = 0.75   # primary task
LANGUAGE_LOSS_W     = 0.25   # auxiliary task

# ── Paths ─────────────────────────────────────────────────
DRIVE_ROOT          = "/content/drive/MyDrive/multilingual_abuse_detection"
MODEL_DIR           = f"{DRIVE_ROOT}/model"
CKPT_DIR            = f"{MODEL_DIR}/checkpoints"
BEST_DIR            = f"{MODEL_DIR}/best_model"
VAL_DIR             = f"{DRIVE_ROOT}/validation"
VAL_PLOTS_DIR       = f"{VAL_DIR}/plots"
VAL_METRICS_DIR     = f"{VAL_DIR}/metrics"
TRAIN_DIR           = f"{DRIVE_ROOT}/training"
TRAIN_LOGS_DIR      = f"{TRAIN_DIR}/logs"

# ── Misc ──────────────────────────────────────────────────
SEED                = 42
DEVICE              = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LANG_TO_IDX         = {"english": 0, "hinglish": 1, "banglish": 2}
IDX_TO_LANG         = {v: k for k, v in LANG_TO_IDX.items()}

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Loss weights — offensive: {OFFENSIVE_LOSS_W}, language: {LANGUAGE_LOSS_W}")


## Section 3 — Drive Mount + Folder Structure

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 3 — DRIVE MOUNT + ORGANISED FOLDER SETUP
#
# Drive layout:
#   multilingual_abuse_detection/
#   ├── model/
#   │   ├── checkpoints/       ← resume states (optimizer, scheduler, scaler)
#   │   └── best_model/        ← best weights by val F1
#   ├── validation/
#   │   ├── plots/             ← all evaluation figures
#   │   └── metrics/           ← CSV / JSON metric tables
#   └── training/
#       └── logs/              ← per-epoch training history CSV
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive")

for d in [CKPT_DIR, BEST_DIR, VAL_PLOTS_DIR, VAL_METRICS_DIR, TRAIN_LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted and folder structure ready ✅")
print(f"  {DRIVE_ROOT}/")
print(f"  ├── model/checkpoints/")
print(f"  ├── model/best_model/")
print(f"  ├── validation/plots/")
print(f"  ├── validation/metrics/")
print(f"  └── training/logs/")


## Section 4 — Checkpoint Manager

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 4 — CHECKPOINT MANAGER
# Persists full training state to Drive so sessions can
# be closed and resumed with zero manual work.
# ═══════════════════════════════════════════════════════════

class CheckpointManager:
    def __init__(self):
        self.ckpt_dir  = Path(CKPT_DIR)
        self.best_dir  = Path(BEST_DIR)
        self.meta_path = self.ckpt_dir / "meta.json"
        self.state_pt  = self.ckpt_dir / "training_state.pt"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.best_dir.mkdir(parents=True, exist_ok=True)

    # ── Save current epoch ────────────────────────────────
    def save(self, model, tokenizer, optimizer, scheduler,
             scaler, epoch, best_f1, es_counter, threshold=0.5):
        model.backbone.save_pretrained(self.ckpt_dir)
        tokenizer.save_pretrained(self.ckpt_dir)
        # Save custom heads separately
        torch.save({
            "offensive_head": model.offensive_head.state_dict(),
            "language_head":  model.language_head.state_dict(),
            "dropout":        model.dropout.state_dict(),
        }, self.ckpt_dir / "heads.pt")
        torch.save({
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler":    scaler.state_dict(),
        }, self.state_pt)
        meta = {
            "epoch":      epoch,
            "best_f1":    float(best_f1),
            "es_counter": es_counter,
            "threshold":  float(threshold),
            "saved_at":   datetime.datetime.now().isoformat(),
        }
        self.meta_path.write_text(json.dumps(meta, indent=2))
        print(f"  💾 Checkpoint — epoch {epoch+1} | best F1 {best_f1:.4f}")

    # ── Save best ─────────────────────────────────────────
    def save_best(self, model, tokenizer):
        model.backbone.save_pretrained(self.best_dir)
        tokenizer.save_pretrained(self.best_dir)
        torch.save({
            "offensive_head": model.offensive_head.state_dict(),
            "language_head":  model.language_head.state_dict(),
            "dropout":        model.dropout.state_dict(),
        }, self.best_dir / "heads.pt")
        print(f"  🏆 Best model saved → {self.best_dir}")

    # ── Resume ────────────────────────────────────────────
    def load(self, model, optimizer, scheduler, scaler):
        if not self.meta_path.exists():
            print("🆕 No checkpoint — fresh training.")
            return 0, 0.0, 0, 0.5

        meta = json.loads(self.meta_path.read_text())

        from transformers import AutoModel
        bb = AutoModel.from_pretrained(str(self.ckpt_dir))
        model.backbone.load_state_dict(bb.state_dict(), strict=False)
        del bb

        heads = torch.load(self.ckpt_dir / "heads.pt", map_location=DEVICE)
        model.offensive_head.load_state_dict(heads["offensive_head"])
        model.language_head.load_state_dict(heads["language_head"])
        model.dropout.load_state_dict(heads["dropout"])

        ts = torch.load(self.state_pt, map_location=DEVICE)
        optimizer.load_state_dict(ts["optimizer"])
        scheduler.load_state_dict(ts["scheduler"])
        scaler.load_state_dict(ts["scaler"])

        print(f"✅ Resumed epoch {meta['epoch']+2} | best F1 {meta['best_f1']:.4f}")
        return meta["epoch"]+1, meta["best_f1"], meta["es_counter"], meta["threshold"]

    def load_best(self, model, tokenizer):
        from transformers import AutoModel, AutoTokenizer
        bb  = AutoModel.from_pretrained(str(self.best_dir))
        model.backbone.load_state_dict(bb.state_dict())
        del bb
        heads = torch.load(self.best_dir / "heads.pt", map_location=DEVICE)
        model.offensive_head.load_state_dict(heads["offensive_head"])
        model.language_head.load_state_dict(heads["language_head"])
        print("✅ Best model weights loaded.")

    def has_checkpoint(self): return self.meta_path.exists()
    def has_best(self):       return (self.best_dir / "heads.pt").exists()


ckpt = CheckpointManager()
print("CheckpointManager ready ✅")


## Section 5 — Data Upload + Balanced Merging

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 5 — DATA UPLOAD
# Upload all 3 TSV files at once.
# Each file must have columns: text | offensive | language
# The notebook auto-identifies which file is which via the
# 'language' column and builds a balanced merged dataset.
#
# Balancing strategy:
#   • Find the smallest dataset  → size = MIN_SIZE
#   • Take ALL rows from the smallest dataset
#   • Sample MIN_SIZE rows from each of the other two
#     (stratified: preserves ~50/50 offensive split)
#   • Total = 3 × MIN_SIZE
# ═══════════════════════════════════════════════════════════
from google.colab import files as colab_files
import io

print("📁 Upload your 3 TSV files (English, Hinglish, Banglish).")
print("   You can select all 3 at once in the file picker.\n")

uploaded = colab_files.upload()   # ← select all 3 files here

# ── Parse uploaded files ──────────────────────────────────
raw_dfs = {}
REQUIRED_COLS = {"text", "offensive", "language"}

for fname, raw_bytes in uploaded.items():
    try:
        df = pd.read_csv(io.BytesIO(raw_bytes), sep="\t")
        df.columns = df.columns.str.strip().str.lower()

        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            print(f"  ⚠️  {fname} missing columns: {missing} — skipped")
            continue

        df["text"]      = df["text"].astype(str).str.strip()
        df["offensive"] = df["offensive"].astype(int)
        df["language"]  = df["language"].astype(str).str.strip().str.lower()

        # Detect which language this file represents
        lang_val = df["language"].mode()[0]
        if lang_val not in LANG_TO_IDX:
            print(f"  ⚠️  {fname} has unknown language '{lang_val}' — skipped")
            continue

        raw_dfs[lang_val] = df
        off_pct = df["offensive"].mean() * 100
        print(f"  ✅ {fname}  →  [{lang_val}]  {len(df):,} rows  |  "
              f"offensive: {off_pct:.1f}%")

    except Exception as e:
        print(f"  ❌ {fname} failed to parse: {e}")

if len(raw_dfs) != 3:
    raise ValueError(f"Expected 3 datasets, got {len(raw_dfs)}: {list(raw_dfs.keys())}")

# ── Balanced sampling ─────────────────────────────────────
def stratified_sample(df, n, seed=SEED):
    """Sample n rows while preserving the offensive/clean ratio."""
    if n >= len(df):
        return df.copy()
    # split by label, sample proportionally
    n_off  = min(int(round(n * 0.5)), (df["offensive"] == 1).sum())
    n_cln  = n - n_off
    off_s  = df[df["offensive"] == 1].sample(n=n_off,  random_state=seed, replace=False)
    cln_s  = df[df["offensive"] == 0].sample(n=n_cln,  random_state=seed, replace=False)
    return pd.concat([off_s, cln_s]).sample(frac=1, random_state=seed).reset_index(drop=True)

sizes      = {lang: len(df) for lang, df in raw_dfs.items()}
min_lang   = min(sizes, key=sizes.get)
MIN_SIZE   = sizes[min_lang]

print(f"\n📊 Dataset sizes: {sizes}")
print(f"   Smallest: [{min_lang}] — {MIN_SIZE:,} rows")
print(f"   Target total: {3 * MIN_SIZE:,} rows (3 × {MIN_SIZE:,})\n")

balanced_parts = []
for lang, df in raw_dfs.items():
    if lang == min_lang:
        part = df.copy()
        label = "all"
    else:
        part  = stratified_sample(df, MIN_SIZE)
        label = f"sampled {MIN_SIZE:,}"
    balanced_parts.append(part)
    off_pct = part["offensive"].mean() * 100
    print(f"  [{lang}]  {label}  →  {len(part):,} rows  |  offensive: {off_pct:.1f}%")

df_all = pd.concat(balanced_parts, ignore_index=True)
df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)
df_all["lang_id"] = df_all["language"].map(LANG_TO_IDX)

print(f"\n✅ Merged dataset: {len(df_all):,} rows")
print(f"   Overall offensive rate: {df_all['offensive'].mean()*100:.1f}%")
print(df_all["language"].value_counts().to_string())


## Section 6 — EDA

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 6 — EXPLORATORY DATA ANALYSIS
# ═══════════════════════════════════════════════════════════
df_all["text_len"] = df_all["text"].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Exploratory Data Analysis", fontsize=14, fontweight="bold")

# Offensive distribution by language
lang_off = df_all.groupby("language")["offensive"].mean() * 100
axes[0,0].bar(lang_off.index, lang_off.values,
              color=["steelblue","salmon","gold"], edgecolor="white")
axes[0,0].set_title("Offensive % by Language")
axes[0,0].set_ylabel("% Offensive")
axes[0,0].set_ylim(0, 100)
for i, (lang, v) in enumerate(lang_off.items()):
    axes[0,0].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")
axes[0,0].grid(axis="y", alpha=0.3)

# Language distribution pie
lang_counts = df_all["language"].value_counts()
axes[0,1].pie(lang_counts.values, labels=lang_counts.index,
              autopct="%1.1f%%", colors=["steelblue","salmon","gold"])
axes[0,1].set_title("Language Distribution")

# Text length distribution by language
for lang in df_all["language"].unique():
    grp = df_all[df_all["language"] == lang]
    axes[1,0].hist(grp["text_len"].clip(0, 200), bins=40,
                   alpha=0.55, label=lang, density=True)
axes[1,0].set_title("Text Length by Language")
axes[1,0].set_xlabel("Word count")
axes[1,0].set_ylabel("Density")
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# Text length: offensive vs clean
for lbl, color in [(0, "steelblue"), (1, "firebrick")]:
    grp = df_all[df_all["offensive"] == lbl]
    axes[1,1].hist(grp["text_len"].clip(0, 200), bins=40,
                   alpha=0.6, color=color,
                   label=f"{'Offensive' if lbl else 'Clean'}", density=True)
axes[1,1].set_title("Text Length: Offensive vs Clean")
axes[1,1].set_xlabel("Word count")
axes[1,1].set_ylabel("Density")
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/eda.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📏 Text length stats by language:")
print(df_all.groupby("language")["text_len"].describe().round(1).to_string())


## Section 7 — Preprocessing

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 7 — PREPROCESSING
# Minimal but multilingual-aware pipeline.
# XLM-R handles most tokenisation, so we only:
#   • Strip URLs / HTML tags
#   • Demojize (emoji → text token)
#   • Collapse repeated characters (haaate → haate)
#   • Expand common English + Hinglish/Banglish abbreviations
#   • Lowercase
# ═══════════════════════════════════════════════════════════
import emoji
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

ENG_ABBREVS = {
    "wtf": "what the fuck",
    "stfu": "shut the fuck up",
    "idk": "I don't know",
    "ngl": "not gonna lie",
    "lmao": "laughing my ass off",
    "omg": "oh my god",
    "u": "you",
    "ur": "your",
    "r": "are",
    "bc": "because",
    "tbh": "to be honest",
    "smh": "shaking my head",
    "af": "as fuck",
    "nvm": "never mind",
    "lol": "laugh out loud",
    "rofl": "rolling on the floor laughing",
    "wbu": "what about you",
    "brb": "be right back",
    "gtg": "got to go",
    "ttyl": "talk to you later",
    "fyi": "for your information",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "btw": "by the way",
    "np": "no problem",
    "ty": "thank you",
    "wyd": "what you doing",
    "hmu": "hit me up",
    "iykyk": "if you know you know",
    "rn": "right now",
    "ikr": "I know right",
    "jk": "just kidding",
    "sfw": "safe for work",
    "nsfw": "not safe for work"
}

DESI_ABBREVS = {
    # Hinglish
    "yaar": "friend",
    "bhai": "brother",
    "abbe": "hey",
    "sala": "jerk",
    "saale": "jerk",
    "bakwas": "nonsense",
    "chup": "silent",
    "aukaat": "worth",
    "ganda": "dirty",
    "bura": "bad",
    "mast": "awesome",
    "kya": "what",
    "toh": "then",
    "nahi": "no",
    "bc": "behanchod",
    "bsdk": "bahenchod ka duffer",
    "mc": "madarchod",
    "bhen": "sister",
    "behen": "sister",
    "bhenchod": "sisterfucker",
    "chutiya": "idiot",
    "gaandu": "asshole",
    "jatt": "cool",
    "jigri": "close friend",
    "panga": "trouble",
    "dimaag": "brain",
    "pakao": "boring",
    "tharki": "pervert",
    "chamcha": "sycophant",
    "kamina": "scoundrel",
    "lauda": "penis",
    "chut": "vagina",
    "lund": "penis",
    # Banglish
    "vai": "brother",
    "apu": "sister",
    "pagla": "crazy",
    "baje": "bad",
    "dhor": "catch",
    "boro": "big",
    "choto": "small",
    "khub": "very",
    "jibon": "life",
    "bhalo": "good",
    "kharap": "bad",
    "ki": "what",
    "keno": "why",
    "kothay": "where",
    "kobe": "when",
    "kemon": "how",
    "tui": "you",
    "tumi": "you (formal)",
    "apni": "you (very formal)",
    "amar": "my",
    "tomar": "your",
    "tor": "your",
    "ei": "this",
    "sei": "that",
    "kintu": "but",
    "tobe": "then",
    "ar": "and",
    "o": "also",
    "na": "no",
    "haoa": "to be",
    "thako": "stay",
    "jao": "go",
    "aslo": "came",
    "dibo": "will give",
    "nibo": "will take",
    "koro": "do",
    "kris": "did",
    "korchis": "doing",
    "korbos": "will do"
}

def clean(text):
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def pipeline(args):
    text, lang = args
    text = clean(str(text))
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)   # repeated chars
    abbrevs = dict(ENG_ABBREVS)
    if lang in ("hinglish", "banglish"):
        abbrevs.update(DESI_ABBREVS)
    tokens = text.lower().split()
    text = " ".join(abbrevs.get(t, t) for t in tokens)
    return text.strip()

print(f"CPU cores available: {cpu_count()}")
args = list(zip(df_all["text"].tolist(), df_all["language"].tolist()))

with Pool(cpu_count()) as pool:
    cleaned = list(tqdm(pool.imap(pipeline, args),
                        total=len(df_all), desc="Preprocessing"))

df_all["clean_text"] = cleaned

print("\nSample before/after:")
for _, row in df_all.sample(5, random_state=42).iterrows():
    print(f"  [{row['language']}] IN : {str(row['text'])[:90]}")
    print(f"             OUT: {row['clean_text'][:90]}\n")


## Section 8 — Train / Val Split

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 8 — STRATIFIED TRAIN / VAL SPLIT
# Stratified by language × offensive label combo so every
# cell of the cross-table is represented in both splits.
# ═══════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

# Create combined stratification key
df_all["strat_key"] = df_all["language"] + "_" + df_all["offensive"].astype(str)

train_df, val_df = train_test_split(
    df_all, test_size=0.1, random_state=SEED,
    stratify=df_all["strat_key"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print("\nTrain offensive % by language:")
print((train_df.groupby("language")["offensive"].mean() * 100).round(1).to_string())
print("\nVal offensive % by language:")
print((val_df.groupby("language")["offensive"].mean() * 100).round(1).to_string())


## Section 9 — Dataset + DataLoaders

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 9 — DATASET + TOKENIZER
# ═══════════════════════════════════════════════════════════
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

print("Loading XLM-RoBERTa-Large tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AbuseDataset(Dataset):
    def __init__(self, df):
        self.texts     = df["clean_text"].tolist()
        self.offensive = df["offensive"].values.astype(np.float32)
        self.lang_ids  = df["lang_id"].values.astype(np.int64)
        self.languages = df["language"].tolist()   # kept for per-lang eval

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length     = MAX_LEN,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "offensive":      torch.tensor(self.offensive[idx], dtype=torch.float),
            "lang_id":        torch.tensor(self.lang_ids[idx],  dtype=torch.long),
            "language":       self.languages[idx],
        }

train_ds = AbuseDataset(train_df)
val_ds   = AbuseDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=2)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Sample input shape: {train_ds[0]['input_ids'].shape}")


## Section 10 — Multi-Task Model (XLM-RoBERTa-Large)

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 10 — MULTI-TASK MODEL
#
# Architecture:
#   XLM-RoBERTa-Large backbone (1024-dim, shared)
#       ├── Offensive Head  → sigmoid → binary BCE
#       └── Language Head   → softmax → cross-entropy
#
# Total loss = 0.75 × offensive_loss + 0.25 × language_loss
# ═══════════════════════════════════════════════════════════
from transformers import AutoModel, get_linear_schedule_with_warmup

class MultiTaskAbuseDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone       = AutoModel.from_pretrained(MODEL_NAME)
        hidden              = self.backbone.config.hidden_size   # 1024
        self.dropout        = nn.Dropout(0.1)
        self.offensive_head = nn.Linear(hidden, 1)
        self.language_head  = nn.Linear(hidden, NUM_LANGUAGES)

    def forward(self, input_ids, attention_mask, output_attentions=False):
        out = self.backbone(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            output_attentions = output_attentions,
        )
        cls = self.dropout(out.last_hidden_state[:, 0, :])   # CLS token
        return {
            "offensive_logits": self.offensive_head(cls),     # (B, 1)
            "language_logits":  self.language_head(cls),      # (B, 3)
            "attentions":       out.attentions if output_attentions else None,
        }


def build_model():
    return MultiTaskAbuseDetector().to(DEVICE)


# ── Build or load from drive ───────────────────────────────
model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built — {n_params:,} parameters")

# ── Class weight for imbalanced offensive label ────────────
pos = max(train_df["offensive"].sum(), 1)
neg = len(train_df) - pos
pos_w = torch.tensor([neg / pos], dtype=torch.float).to(DEVICE)
print(f"Offensive class weight: {pos_w.item():.2f}×")

offensive_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
language_criterion  = nn.CrossEntropyLoss()

# ── Optimizer + Scheduler ─────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
total_steps  = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)
scaler = torch.cuda.amp.GradScaler()

print(f"Total optim steps : {total_steps}")
print(f"Warmup steps      : {warmup_steps}")
print("Optimizer, scheduler, scaler ready ✅")


## Section 11 — Training

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 11 — TRAINING LOOP
# Features:
#   ✓ Auto-resume from Drive on session restart
#   ✓ Gradient accumulation  (effective batch = 32)
#   ✓ Mixed-precision AMP    (2-3× faster on T4)
#   ✓ Early stopping         (patience = 5 epochs)
#   ✓ Multi-task loss        (offensive + language)
#   ✓ All plots / CSVs saved to organised Drive folders
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score, accuracy_score

# ── Resume if checkpoint exists ────────────────────────────
start_epoch, best_f1, es_counter, best_threshold = ckpt.load(
    model, optimizer, scheduler, scaler
)
model = model.to(DEVICE)

# ── Training history (append-safe) ────────────────────────
history_path = f"{TRAIN_LOGS_DIR}/training_history.csv"
if os.path.exists(history_path):
    history_df = pd.read_csv(history_path)
    history    = history_df.to_dict("records")
else:
    history = []

print(f"Starting epoch {start_epoch+1}/{EPOCHS}")
print(f"Early stop patience: {EARLY_STOP_PATIENCE}\n")

for epoch in range(start_epoch, EPOCHS):

    # ─── TRAIN ─────────────────────────────────────────────
    model.train()
    total_loss = off_loss_sum = lang_loss_sum = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Ep {epoch+1}/{EPOCHS} [Train]")

    for step, batch in pbar:
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        off  = batch["offensive"].to(DEVICE, non_blocking=True)
        lid  = batch["lang_id"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            out      = model(input_ids=ids, attention_mask=mask)
            off_loss = offensive_criterion(out["offensive_logits"].squeeze(-1), off)
            lng_loss = language_criterion(out["language_logits"], lid)
            loss     = (OFFENSIVE_LOSS_W * off_loss + LANGUAGE_LOSS_W * lng_loss)
            loss     = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step+1) % GRAD_ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        real_loss     = loss.item() * GRAD_ACCUM_STEPS
        total_loss   += real_loss
        off_loss_sum += off_loss.item()
        lang_loss_sum+= lng_loss.item()
        pbar.set_postfix({"loss": f"{real_loss:.4f}",
                          "off": f"{off_loss.item():.4f}",
                          "lng": f"{lng_loss.item():.4f}"})

    avg_total = total_loss   / len(train_loader)
    avg_off   = off_loss_sum / len(train_loader)
    avg_lang  = lang_loss_sum/ len(train_loader)

    # ─── VALIDATE ──────────────────────────────────────────
    model.eval()
    all_probs, all_off_labels = [], []
    all_lang_preds, all_lang_labels = [], []
    all_val_langs = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Val]  "):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.cuda.amp.autocast():
                out = model(input_ids=ids, attention_mask=mask)
            all_probs.extend(torch.sigmoid(out["offensive_logits"].squeeze(-1)).cpu().numpy())
            all_off_labels.extend(batch["offensive"].numpy())
            all_lang_preds.extend(out["language_logits"].argmax(-1).cpu().numpy())
            all_lang_labels.extend(batch["lang_id"].numpy())
            all_val_langs.extend(batch["language"])

    all_probs      = np.array(all_probs)
    all_off_labels = np.array(all_off_labels).astype(int)
    all_lang_preds = np.array(all_lang_preds)
    all_lang_labels= np.array(all_lang_labels)

    # Find best threshold this epoch
    best_t_epoch, best_f1_epoch = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.02):
        preds = (all_probs > t).astype(int)
        f = f1_score(all_off_labels, preds, average="macro", zero_division=0)
        if f > best_f1_epoch:
            best_f1_epoch, best_t_epoch = f, t

    val_preds  = (all_probs > best_t_epoch).astype(int)
    lang_acc   = accuracy_score(all_lang_labels, all_lang_preds)

    # ─── Per-language F1 ───────────────────────────────────
    per_lang_f1 = {}
    for lang, lid in LANG_TO_IDX.items():
        mask = np.array(all_val_langs) == lang
        if mask.sum() < 5: continue
        per_lang_f1[lang] = f1_score(
            all_off_labels[mask], val_preds[mask],
            average="macro", zero_division=0
        )

    # ─── Log ───────────────────────────────────────────────
    rec = {
        "epoch":       epoch+1,
        "train_loss":  round(avg_total, 4),
        "off_loss":    round(avg_off,   4),
        "lang_loss":   round(avg_lang,  4),
        "val_f1_macro":round(best_f1_epoch, 4),
        "val_threshold":round(best_t_epoch, 3),
        "lang_acc":    round(lang_acc, 4),
        **{f"f1_{l}": round(v, 4) for l, v in per_lang_f1.items()},
    }
    history.append(rec)
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(f"\n{'='*58}")
    print(f"  Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss total : {avg_total:.4f}  "
          f"(offensive={avg_off:.4f} | lang={avg_lang:.4f})")
    print(f"  Val F1 (macro)   : {best_f1_epoch:.4f}  "
          f"(threshold={best_t_epoch:.3f})")
    print(f"  Lang ID Accuracy : {lang_acc*100:.1f}%")
    print(f"  ES counter       : {es_counter}/{EARLY_STOP_PATIENCE}")
    for lang, f in per_lang_f1.items():
        print(f"    [{lang}]  F1={f:.4f}")
    print(f"{'='*58}")

    # ─── Save / Early stop ─────────────────────────────────
    if best_f1_epoch > best_f1:
        best_f1       = best_f1_epoch
        best_threshold= best_t_epoch
        es_counter    = 0
        ckpt.save_best(model, tokenizer)
        ckpt.save(model, tokenizer, optimizer, scheduler, scaler,
                  epoch, best_f1, es_counter, best_threshold)
        print(f"  🏆 New best! F1={best_f1:.4f}")
    else:
        es_counter += 1
        ckpt.save(model, tokenizer, optimizer, scheduler, scaler,
                  epoch, best_f1, es_counter, best_threshold)
        print(f"  No improvement. ES={es_counter}/{EARLY_STOP_PATIENCE}")
        if es_counter >= EARLY_STOP_PATIENCE:
            print("\n🛑 EARLY STOPPING triggered.")
            break

# ── Training curve ─────────────────────────────────────────
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], "b-o", markersize=4)
axes[0].set_title("Total Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)
axes[1].plot(hist_df["epoch"], hist_df["val_f1_macro"], "g-o", markersize=4)
axes[1].set_title("Val Macro F1"); axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)
axes[2].plot(hist_df["epoch"], hist_df["lang_acc"]*100, "r-o", markersize=4)
axes[2].set_title("Language ID Accuracy (%)"); axes[2].set_xlabel("Epoch"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n🎉 Training done! Best Val F1 = {best_f1:.4f} | Threshold = {best_threshold:.3f}")


## Section 12 — Load Best Model

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 12 — LOAD BEST MODEL FOR EVALUATION
# Always loads from the best_model/ folder, not last ckpt.
# ═══════════════════════════════════════════════════════════
from transformers import AutoTokenizer as AT

eval_model     = build_model()
eval_tokenizer = AT.from_pretrained(str(ckpt.best_dir)) if ckpt.has_best() \
                 else tokenizer

if ckpt.has_best():
    ckpt.load_best(eval_model, eval_tokenizer)
else:
    print("⚠️  No best model on Drive — using current model weights.")
    eval_model.load_state_dict(model.state_dict())

eval_model.eval()
print("Best model loaded ✅")

# ── Load saved threshold ───────────────────────────────────
if ckpt.has_checkpoint():
    meta = json.loads(ckpt.meta_path.read_text())
    best_threshold = meta.get("threshold", 0.5)
print(f"Using threshold: {best_threshold:.3f}")

# ── Full val predictions ───────────────────────────────────
def get_val_preds(model, loader):
    all_probs, all_off, all_lang_pred, all_lang_true, all_langs = [], [], [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.cuda.amp.autocast():
                out = model(input_ids=ids, attention_mask=mask)
            all_probs.extend(torch.sigmoid(out["offensive_logits"].squeeze(-1)).cpu().numpy())
            all_off.extend(batch["offensive"].numpy())
            all_lang_pred.extend(out["language_logits"].argmax(-1).cpu().numpy())
            all_lang_true.extend(batch["lang_id"].numpy())
            all_langs.extend(batch["language"])
    return (np.array(all_probs), np.array(all_off).astype(int),
            np.array(all_lang_pred), np.array(all_lang_true), all_langs)

val_probs, val_off_labels, val_lang_pred, val_lang_true, val_langs = \
    get_val_preds(eval_model, val_loader)
val_preds = (val_probs > best_threshold).astype(int)
print(f"Val predictions shape: {val_probs.shape}")


## Section 13 — Threshold Tuning

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 13 — THRESHOLD TUNING
# Sweeps thresholds 0.05 → 0.95 and picks the one that
# maximises macro F1 on the validation set.
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score as sk_f1, precision_score, recall_score

thresholds = np.arange(0.05, 0.95, 0.01)
f1s, precs, recs = [], [], []

for t in thresholds:
    p = (val_probs > t).astype(int)
    f1s.append(sk_f1(val_off_labels, p, average="macro", zero_division=0))
    precs.append(precision_score(val_off_labels, p, zero_division=0))
    recs.append(recall_score(val_off_labels, p, zero_division=0))

best_idx      = int(np.argmax(f1s))
best_threshold= float(thresholds[best_idx])
val_preds     = (val_probs > best_threshold).astype(int)

# Save to Drive meta
if ckpt.has_checkpoint():
    meta = json.loads(ckpt.meta_path.read_text())
    meta["threshold"] = best_threshold
    ckpt.meta_path.write_text(json.dumps(meta, indent=2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1s,   "g-",  label="Macro F1",   linewidth=2)
ax.plot(thresholds, precs, "b--", label="Precision",  linewidth=1.5)
ax.plot(thresholds, recs,  "r--", label="Recall",     linewidth=1.5)
ax.axvline(best_threshold, color="black", linestyle=":", linewidth=2,
           label=f"Best t={best_threshold:.2f} | F1={f1s[best_idx]:.4f}")
ax.scatter([best_threshold], [f1s[best_idx]], color="black", s=80, zorder=5)
ax.set_title("Threshold Sweep — Offensive Detection")
ax.set_xlabel("Threshold"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n🎯 Optimal threshold : {best_threshold:.3f}")
print(f"   F1 (macro)        : {f1s[best_idx]:.4f}")
print(f"   Precision         : {precs[best_idx]:.4f}")
print(f"   Recall            : {recs[best_idx]:.4f}")


## Section 14 — Full Evaluation Suite

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14A — OVERALL OFFENSIVE DETECTION METRICS
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, roc_curve,
    average_precision_score, matthews_corrcoef,
    accuracy_score, hamming_loss, f1_score,
)

yt = val_off_labels
yp = val_preds
yprob = val_probs

print("=" * 60)
print("OFFENSIVE DETECTION — OVERALL METRICS")
print("=" * 60)
metrics = {
    "Accuracy":         round(accuracy_score(yt, yp), 4),
    "Macro F1":         round(f1_score(yt, yp, average="macro",    zero_division=0), 4),
    "Micro F1":         round(f1_score(yt, yp, average="micro",    zero_division=0), 4),
    "Weighted F1":      round(f1_score(yt, yp, average="weighted", zero_division=0), 4),
    "ROC-AUC":          round(roc_auc_score(yt, yprob), 4),
    "Avg Precision":    round(average_precision_score(yt, yprob), 4),
    "MCC":              round(matthews_corrcoef(yt, yp), 4),
}
for k, v in metrics.items():
    bar = "█" * int(abs(v) * 20)
    print(f"  {k:<20}: {v:.4f}  [{bar:<20}]")

print("\n" + classification_report(yt, yp,
      target_names=["Clean","Offensive"], zero_division=0))

# Save metrics JSON
with open(f"{VAL_METRICS_DIR}/offensive_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to Drive ✅")


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14B — LANGUAGE IDENTIFICATION METRICS
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import classification_report as cr

print("=" * 60)
print("LANGUAGE IDENTIFICATION METRICS")
print("=" * 60)
lang_names = [IDX_TO_LANG[i] for i in range(NUM_LANGUAGES)]
print(cr(val_lang_true, val_lang_pred,
         target_names=lang_names, zero_division=0))

lang_id_acc = accuracy_score(val_lang_true, val_lang_pred)
print(f"Overall Language ID Accuracy: {lang_id_acc*100:.2f}%")

# Confusion matrix — language
fig, ax = plt.subplots(figsize=(6, 5))
cm_lang = confusion_matrix(val_lang_true, val_lang_pred)
sns.heatmap(cm_lang, annot=True, fmt="d", ax=ax, cmap="Blues",
            xticklabels=lang_names, yticklabels=lang_names)
ax.set_title("Language ID Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/lang_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14C — PER-LANGUAGE OFFENSIVE DETECTION BREAKDOWN
# ═══════════════════════════════════════════════════════════
val_lang_arr = np.array(val_langs)
lang_results = {}

print("=" * 60)
print("PER-LANGUAGE OFFENSIVE DETECTION")
print("=" * 60)

for lang in LANG_TO_IDX:
    mask = val_lang_arr == lang
    if mask.sum() < 10: continue
    lt = val_off_labels[mask]
    lp = val_preds[mask]
    lb = val_probs[mask]

    macro = f1_score(lt, lp, average="macro",    zero_division=0)
    micro = f1_score(lt, lp, average="micro",    zero_division=0)
    wt    = f1_score(lt, lp, average="weighted", zero_division=0)
    try:  auc = roc_auc_score(lt, lb)
    except: auc = float("nan")
    mcc   = matthews_corrcoef(lt, lp) if lt.sum() > 0 else 0.0
    acc   = accuracy_score(lt, lp)

    lang_results[lang] = {"n": int(mask.sum()), "macro_f1": macro,
                          "micro_f1": micro, "auc": auc}
    print(f"\n  [{lang.upper()}]  n={mask.sum():,}")
    print(f"    Accuracy     : {acc:.4f}")
    print(f"    Macro F1     : {macro:.4f}")
    print(f"    Micro F1     : {micro:.4f}")
    print(f"    Weighted F1  : {wt:.4f}")
    print(f"    ROC-AUC      : {auc:.4f}" if not np.isnan(auc) else "    ROC-AUC: n/a")
    print(f"    MCC          : {mcc:.4f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
langs_l  = list(lang_results.keys())
colors   = ["steelblue","salmon","gold"]

axes[0].bar(langs_l, [lang_results[l]["macro_f1"] for l in langs_l],
            color=colors, edgecolor="white")
axes[0].set_title("Macro F1 by Language"); axes[0].set_ylim(0,1)
axes[0].grid(axis="y", alpha=0.3)
for i, (l, v) in enumerate([(l, lang_results[l]["macro_f1"]) for l in langs_l]):
    axes[0].text(i, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")

axes[1].bar(langs_l, [lang_results[l]["auc"] for l in langs_l],
            color=colors, edgecolor="white")
axes[1].set_title("ROC-AUC by Language"); axes[1].set_ylim(0,1)
axes[1].grid(axis="y", alpha=0.3)
for i, (l, v) in enumerate([(l, lang_results[l]["auc"]) for l in langs_l]):
    if not np.isnan(v):
        axes[1].text(i, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/per_language_offensive.png", dpi=150, bbox_inches="tight")
plt.show()

# Save
with open(f"{VAL_METRICS_DIR}/per_language_metrics.json", "w") as f:
    json.dump(lang_results, f, indent=2)


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14D — ROC + PRECISION-RECALL CURVES
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

fpr, tpr, _ = roc_curve(val_off_labels, val_probs)
auc_val     = roc_auc_score(val_off_labels, val_probs)
axes[0].plot(fpr, tpr, "b-", linewidth=2, label=f"Overall (AUC={auc_val:.3f})")

for lang, color in zip(LANG_TO_IDX, ["steelblue","salmon","gold"]):
    mask = val_lang_arr == lang
    if mask.sum() < 10 or val_off_labels[mask].sum() == 0: continue
    fpr_l, tpr_l, _ = roc_curve(val_off_labels[mask], val_probs[mask])
    auc_l = roc_auc_score(val_off_labels[mask], val_probs[mask])
    axes[0].plot(fpr_l, tpr_l, "--", color=color,
                 linewidth=1.5, label=f"{lang} ({auc_l:.3f})")

axes[0].plot([0,1],[0,1],"k--", alpha=0.4)
axes[0].set_title("ROC Curves"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

prec_c, rec_c, _ = precision_recall_curve(val_off_labels, val_probs)
ap = average_precision_score(val_off_labels, val_probs)
axes[1].plot(rec_c, prec_c, "b-", linewidth=2, label=f"Overall (AP={ap:.3f})")

for lang, color in zip(LANG_TO_IDX, ["steelblue","salmon","gold"]):
    mask = val_lang_arr == lang
    if mask.sum() < 10 or val_off_labels[mask].sum() == 0: continue
    pc, rc, _ = precision_recall_curve(val_off_labels[mask], val_probs[mask])
    ap_l = average_precision_score(val_off_labels[mask], val_probs[mask])
    axes[1].plot(rc, pc, "--", color=color,
                 linewidth=1.5, label=f"{lang} ({ap_l:.3f})")

axes[1].set_title("Precision-Recall Curves"); axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14E — CONFUSION MATRIX + CALIBRATION
# ═══════════════════════════════════════════════════════════
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(val_off_labels, val_preds)
sns.heatmap(cm, annot=True, fmt="d", ax=axes[0], cmap="Blues",
            xticklabels=["Clean","Offensive"],
            yticklabels=["Clean","Offensive"])
tn, fp, fn, tp = cm.ravel()
axes[0].set_title(
    f"Confusion Matrix\nTPR={tp/(tp+fn+1e-9):.3f}  FPR={fp/(fp+tn+1e-9):.3f}"
)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

# Calibration
try:
    frac, mean_pred = calibration_curve(val_off_labels, val_probs,
                                         n_bins=10, strategy="quantile")
    axes[1].plot(mean_pred, frac, "s-b", linewidth=1.5, label="Model")
    axes[1].plot([0,1],[0,1],"k--", label="Perfect")
    axes[1].fill_between(mean_pred, frac, mean_pred, alpha=0.15, color="red")
    axes[1].set_title("Calibration Diagram")
    axes[1].set_xlabel("Mean predicted prob")
    axes[1].set_ylabel("Fraction of positives")
    axes[1].legend(); axes[1].grid(alpha=0.3)
except Exception as e:
    axes[1].text(0.5, 0.5, str(e), ha="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/confusion_calibration.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14F — CONFIDENCE HISTOGRAM + ERROR ANALYSIS
# ═══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(val_probs[val_off_labels == 0], bins=50, alpha=0.6,
        color="steelblue", label="Clean", density=True)
ax.hist(val_probs[val_off_labels == 1], bins=50, alpha=0.6,
        color="firebrick", label="Offensive", density=True)
ax.axvline(best_threshold, color="black", linestyle="--", linewidth=2,
           label=f"Threshold = {best_threshold:.2f}")
ax.set_title("Confidence Distribution")
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Density")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{VAL_PLOTS_DIR}/confidence_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Error Analysis ─────────────────────────────────────────
val_text_arr = np.array(val_df["clean_text"].tolist())
fp_mask = (val_preds == 1) & (val_off_labels == 0)
fn_mask = (val_preds == 0) & (val_off_labels == 1)
tp_mask = (val_preds == 1) & (val_off_labels == 1)

print(f"\n{'='*60}")
print(f"ERROR ANALYSIS")
print(f"  FP={fp_mask.sum()} | FN={fn_mask.sum()} | TP={tp_mask.sum()}")
print(f"{'='*60}")

print("\n🔴 TOP 5 FALSE POSITIVES (predicted offensive, actually clean):")
fps = sorted(zip(val_probs[fp_mask], val_text_arr[fp_mask]), reverse=True)[:5]
for prob, text in fps:
    print(f"  [{prob:.3f}] {text[:110]}")

print("\n🟡 TOP 5 FALSE NEGATIVES (offensive content missed):")
fns = sorted(zip(val_probs[fn_mask], val_text_arr[fn_mask]))[:5]
for prob, text in fns:
    print(f"  [{prob:.3f}] {text[:110]}")

print("\n🟢 TOP 5 CONFIDENT TRUE POSITIVES:")
tps = sorted(zip(val_probs[tp_mask], val_text_arr[tp_mask]), reverse=True)[:5]
for prob, text in tps:
    print(f"  [{prob:.3f}] {text[:110]}")

# Save full metrics CSV
summary = pd.DataFrame([{
    "metric": k, "value": v
} for k, v in metrics.items()])
summary.to_csv(f"{VAL_METRICS_DIR}/summary_metrics.csv", index=False)
print("\n✅ All evaluation plots and metrics saved to Drive.")


## Section 15 — Attention Visualization

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 15 — ATTENTION VISUALIZATION
# Extracts CLS-row attention from the last transformer layer
# averaged over all heads — shows which tokens the model
# focused on when making its prediction.
# ═══════════════════════════════════════════════════════════
from langdetect import detect, LangDetectException
def predict_with_attention(text):
    """Returns offensive prob, language prediction, word attention."""

    try:
        detected = detect(str(text)[:300])
        lang_hint = "hinglish" if detected == "hi" else "english"
    except LangDetectException:
        lang_hint = "english"

    cleaned = pipeline((text, lang_hint))
    inputs  = eval_tokenizer(
        cleaned,
        return_tensors = "pt",
        max_length     = MAX_LEN,
        truncation     = True,
        padding        = True,
    ).to(DEVICE)

    with torch.no_grad():
        out = eval_model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            output_attentions = True,
        )

    off_prob  = float(torch.sigmoid(out["offensive_logits"].squeeze(-1)).cpu())
    lang_pred = int(out["language_logits"].argmax(-1).cpu())

    # Last-layer attention, avg heads, CLS row
    if out["attentions"] is None:
        print("Warning: Attention weights are not available for visualization. Skipping attention plot.")
        return off_prob, IDX_TO_LANG[lang_pred], [], np.array([])

    attn    = out["attentions"][-1][0].mean(0)[0].cpu().numpy()
    real_n  = int((inputs["input_ids"][0] != eval_tokenizer.pad_token_id).sum())
    tokens  = eval_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[:real_n]
    attn    = attn[:real_n]
    attn   /= (attn.max() + 1e-9)

    # Merge sub-word tokens
    words, wts = [], []
    for tok_s, w in zip(tokens, attn):
        clean_t = tok_s.replace("▁","").replace("##","")
        if clean_t in ("<s>","</s>","[CLS]","[SEP]","[PAD]",""): continue
        if words and (tok_s.startswith("##") or not tok_s.startswith("▁")):
            words[-1] += clean_t; wts[-1] = max(wts[-1], float(w))
        else:
            words.append(clean_t); wts.append(float(w))

    return off_prob, IDX_TO_LANG[lang_pred], words[:24], np.array(wts[:24])


def plot_prediction(text):
    off_prob, lang_pred, words, wts = predict_with_attention(text)
    verdict    = "🔴 OFFENSIVE" if off_prob > best_threshold else "🟢 CLEAN"
    confidence = off_prob if off_prob > best_threshold else 1 - off_prob

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4),
                                    gridspec_kw={"width_ratios": [3, 1]})
    fig.suptitle(
        f'"{text[:90]}{"…" if len(text)>90 else ""}"',
        fontsize=9, style="italic"
    )

    if len(words):
        norm = wts / (wts.max() + 1e-9)
        bars = ax1.bar(range(len(words)), norm, color=plt.cm.Reds(norm), edgecolor="white")
        ax1.set_xticks(range(len(words)))
        ax1.set_xticklabels(words, rotation=40, ha="right", fontsize=8)
    ax1.set_ylabel("Attention (normalised)")
    ax1.set_title("Word Attention — Last Layer, Avg Heads")
    ax1.grid(axis="y", alpha=0.3)

    color = "firebrick" if off_prob > best_threshold else "steelblue"
    ax2.barh(["Offensive","Clean"],
             [off_prob, 1 - off_prob],
             color=[color, "steelblue" if color=="firebrick" else "firebrick"],
             edgecolor="white")
    ax2.axvline(best_threshold, color="gray", linestyle=":", linewidth=1.5)
    ax2.set_xlim(0, 1)
    ax2.set_title(f"Prediction\n{verdict}\nLang: {lang_pred}")
    ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"  Offensive prob  : {off_prob:.3f}  (threshold={best_threshold:.3f})")
    print(f"  Detected lang   : {lang_pred}")
    print(f"  Verdict         : {verdict}  (confidence {confidence:.1%})")
    return off_prob


# ── Sample visualizations ─────────────────────────────────
examples = [
    "you are such a complete moron, what is wrong with you",
    "yaar tu bahut bura insaan hai bhai seriously",
    "vai tumi onek baje kotha bolo",
    "I hope you have an absolutely wonderful day!",
    "kya bakwas kar raha hai tu saale",
]
for ex in examples:
    print(f"\n{'─'*55}")
    plot_prediction(ex)

## Section 16 — Interactive Testing

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 16 — INTERACTIVE TESTING CELL
# Type any text in English, Hinglish, or Banglish.
# Returns:
#   • Detected language (model prediction)
#   • Offensive probability + verdict
#   • Word attention bar chart
#   • Confidence meter
# ═══════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, clear_output

title = widgets.HTML(
    "<h3 style='color:#2c3e50;margin-bottom:4px'>"
    "🛡️ Multilingual Abuse Detector</h3>"
    "<p style='color:#7f8c8d;margin:0'>"
    "English · Hinglish · Banglish &nbsp;|&nbsp; XLM-RoBERTa-Large</p>"
)
text_box = widgets.Textarea(
    placeholder = "Type a sentence here (any language)…",
    layout      = widgets.Layout(width="100%", height="80px"),
)
btn = widgets.Button(
    description  = "🔍  Analyse",
    button_style = "primary",
    layout       = widgets.Layout(width="130px", height="36px"),
)
clear_btn = widgets.Button(
    description  = "🗑  Clear",
    button_style = "warning",
    layout       = widgets.Layout(width="100px", height="36px"),
)
out = widgets.Output()

def on_analyse(b):
    with out:
        clear_output(wait=True)
        text = text_box.value.strip()
        if not text:
            print("⚠️  Please enter some text."); return

        off_prob, lang_pred, words, wts = predict_with_attention(text)
        verdict = "🔴 OFFENSIVE" if off_prob > best_threshold else "🟢 CLEAN"
        conf    = off_prob if off_prob > best_threshold else 1 - off_prob

        print(f"{'─'*56}")
        print(f"Input    : {text}")
        print(f"Language : {lang_pred.upper()}   (model prediction)")
        print(f"{'─'*56}")

        filled = int(off_prob * 30)
        bar    = "█" * filled + "░" * (30 - filled)
        print(f"Offensive: {off_prob:.3f}  [{bar}]")
        print(f"Threshold: {best_threshold:.3f}  {'▲' * int(best_threshold*30)}")
        print(f"{'─'*56}")
        print(f"Verdict  : {verdict}  (confidence {conf:.1%})")

        # Attention chart
        if words:
            fig, ax = plt.subplots(figsize=(12, 2.8))
            norm    = wts / (wts.max() + 1e-9)
            ax.bar(words, norm, color=plt.cm.Reds(norm), edgecolor="white")
            ax.set_title("Word Attention (which tokens influenced the prediction)",
                         fontsize=10)
            ax.set_ylabel("Attention"); ax.grid(axis="y", alpha=0.3)
            ax.set_xticklabels(words, rotation=35, ha="right", fontsize=9)
            plt.tight_layout(); plt.show()

def on_clear(b):
    with out:
        clear_output()
    text_box.value = ""

btn.on_click(on_analyse)
clear_btn.on_click(on_clear)

display(widgets.VBox([
    title,
    widgets.Label("Enter text (English, Hinglish, or Banglish):"),
    text_box,
    widgets.HBox([btn, clear_btn], layout=widgets.Layout(gap="8px")),
    out,
], layout=widgets.Layout(
    padding        = "18px",
    border         = "1px solid #ddd",
    border_radius  = "10px",
    width          = "820px",
    background_color = "#fafafa",
)))
